# fortls

In [ ]:
# Example 1: the `fortls` command is on PATH and reports the packaged version.
import shutil
import subprocess

import fortls

exe = shutil.which("fortls")
assert exe, "the fortls console script is missing from PATH"
out = subprocess.run([exe, "--version"], capture_output=True, text=True, timeout=120)
assert out.returncode == 0, out.stderr
assert out.stdout.strip() == fortls.__version__, (out.stdout, fortls.__version__)
print(exe, out.stdout.strip())

In [ ]:
# Example 2: fortls' own parser finds a module and the function inside it.
import pathlib
import tempfile

from fortls.parsers.internal.parser import FortranFile

work = pathlib.Path(tempfile.mkdtemp(prefix="fortls-nb-"))
geometry = work / "geometry.f90"
geometry.write_text(
    "module geometry\n  implicit none\ncontains\n  real function area(r)\n"
    "    real, intent(in) :: r\n    area = 3.14159 * r * r\n  end function area\nend module geometry\n"
)
parsed = FortranFile(str(geometry))
error, _ = parsed.load_from_disk()
assert error is None, error
scopes = [(scope.name, type(scope).__name__) for scope in parsed.parse().get_scopes()]
print(scopes)
assert ("geometry", "Module") in scopes and ("area", "Function") in scopes, scopes

In [ ]:
# Example 3: fixed-form (column-based, Fortran 77) vs free-form detection, the cases fortls documents.
from fortls.helper_functions import detect_fixed_format

fixed = ["      PROGRAM MAIN\n", "C     a comment in column 1\n", "      END\n"]
free = [" INTEGER, PARAMETER :: N = 10\n"]
continued = ["trailing line & ! comment\n"]
print("fixed:", detect_fixed_format(fixed), "free:", detect_fixed_format(free), "ampersand:", detect_fixed_format(continued))
assert detect_fixed_format(fixed) is True
assert detect_fixed_format(free) is False
assert detect_fixed_format(continued) is False

In [ ]:
# Example 4: over the language-server protocol, a variable declared twice is reported, and a clean file is not.
import io

from fortls.jsonrpc import ReadWriter, path_to_uri, read_rpc_messages, write_rpc_notification, write_rpc_request

bad = work / "bad.f90"
bad.write_text("program p\n  implicit none\n  integer :: n\n  real :: n\n  n = 1\nend program p\n")
def open_doc(path):
    return write_rpc_notification("textDocument/didOpen", {"textDocument": {"uri": path_to_uri(str(path)), "languageId": "fortran", "version": 1, "text": path.read_text()}})
request = write_rpc_request(1, "initialize", {"rootPath": str(work), "rootUri": path_to_uri(str(work))})
request += open_doc(bad) + open_doc(geometry) + write_rpc_request(2, "shutdown", None) + write_rpc_notification("exit", None)
out = subprocess.run([exe, "--disable_autoupdate"], input=request.encode(), capture_output=True, timeout=120)
messages = read_rpc_messages(ReadWriter(io.BytesIO(out.stdout), None))
diagnostics = {m["params"]["uri"]: m["params"]["diagnostics"] for m in messages if m.get("method") == "textDocument/publishDiagnostics"}
print({uri.rsplit("/", 1)[-1]: [d["message"] for d in found] for uri, found in diagnostics.items()})
assert any('"n" declared twice' in d["message"] for d in diagnostics[path_to_uri(str(bad))])
assert diagnostics[path_to_uri(str(geometry))] == []
assert not any("update the fortls" in str(m) for m in messages), "fortls tried to update itself"

In [ ]:
# Example 5: go-to-definition from a `use` site lands on the function, and the module's symbols are listed.
user = work / "user.f90"
user.write_text("program q\n  use geometry\n  implicit none\n  print *, area(2.0)\nend program q\n")
request = write_rpc_request(1, "initialize", {"rootPath": str(work), "rootUri": path_to_uri(str(work))})
request += write_rpc_request(2, "textDocument/definition", {"textDocument": {"uri": path_to_uri(str(user))}, "position": {"line": 3, "character": 13}})
request += write_rpc_request(3, "textDocument/documentSymbol", {"textDocument": {"uri": path_to_uri(str(geometry))}})
request += write_rpc_request(4, "shutdown", None) + write_rpc_notification("exit", None)
out = subprocess.run([exe, "--disable_autoupdate"], input=request.encode(), capture_output=True, timeout=120)
answers = {m["id"]: m.get("result") for m in read_rpc_messages(ReadWriter(io.BytesIO(out.stdout), None)) if "id" in m}
print("definition:", answers[2])
assert answers[2]["uri"] == path_to_uri(str(geometry)) and answers[2]["range"]["start"]["line"] == 3, answers[2]
symbols = sorted(symbol["name"] for symbol in answers[3])
print("symbols:", symbols)
assert symbols == ["area", "geometry"], symbols